<a href="https://colab.research.google.com/github/andy-mears/DTSC5502-Ensemble_GradientBoosting_NSL-KDD/blob/main/Ensemble_GradientBoosting_NSL_KDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run to install catboost
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.7 MB/s eta 0:00:00


# Import Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np
from collections import Counter

In [ ]:
class EnsembleClassifier(BaseEstimator, ClassifierMixin):
  """
  An ensemble classifier that combines multiple different classifiers
    and handles categorical features appropriately for each.
    """
  def __init__(self, estimators, categorical_features):
    """
    Initializes the ensemble classifier.

    Args:
      estimators: A list of (name, model) pairs. 'name' is a text label for a model
        and 'model' is the actual machine learning algorithm object (e.g., XGBClassifier()).
      categorical_features (list of str): A list of column names in the input data
        that should be treated as categorical features.
    """
    self.estimators = estimators
    self.categorical_features = categorical_features
    self.fitted_estimators_ = {}
    self.label_encoders_ = {}
    self.one_hot_encoders_ = {}

  def fit(self, X, y):
    """
    Fits the ensemble classifier to the training data.

    Args:
      X (pd.DataFrame): The training input features.
      y (pd.Series): The training target labels.

    Returns:
      self: The fitted ensemble classifier.
    """
    # Reset fitted estimators and encoders for a new fit
    self.fitted_estimators_ = {}
    self.label_encoders_ = {}
    self.one_hot_encoders_ = {}
    # Iterate through each estimator in the ensemble
    for name, estimator in self.estimators:
        if name == 'catboost':
            # CatBoost can handle categorical features directly
            estimator.fit(X[self.categorical_features], y, cat_features=self.categorical_features, verbose=0)
            self.fitted_estimators_[name] = estimator
        else:
            #  XGBoost and LightGBM require numerical input
            # LabelEncoder converts categorical labels into numerical labels
            le = LabelEncoder()
            y_encoded = le.fit_transform(y)
            self.label_encoders_[name] = le

            # One-hot encode the categorical features
            ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
            X_categorical = X[self.categorical_features]
            ohe.fit(X_categorical)
            X_encoded_categorical = ohe.transform(X_categorical)
            feature_names = ohe.get_feature_names_out(input_features=self.categorical_features)
            X_encoded_categorical_df = pd.DataFrame(X_encoded_categorical, index=X.index, columns=feature_names)

            # Combine numerical features with the encoded categorical features
            X_numerical = X.drop(columns=self.categorical_features)
            X_encoded = pd.concat([X_numerical, X_encoded_categorical_df], axis=1)

            # Fit the individual estimator on the processed data
            estimator.fit(X_encoded, y_encoded)
            self.fitted_estimators_[name] = estimator
            self.one_hot_encoders_[name] = ohe
    return self

  def predict(self, X):
    """
    Predicts the class labels for the input data.

      Args:
        X (pd.DataFrame): The input features to predict on.

      Returns:
        np.ndarray: The predicted class labels.
    """

    predictions = {}  # Store predictions from each individual estimator
    # Iterate through the fitted estimators
    for name, estimator in self.fitted_estimators_.items():
      if name == 'catboost':
        predictions[name] = estimator.predict(X[self.categorical_features])
      else:
        # For other estimators, preprocess the categorical features as done during training
        ohe = self.one_hot_encoders_[name]
        X_categorical = X[self.categorical_features]
        X_encoded_categorical = ohe.transform(X_categorical)
        feature_names = ohe.get_feature_names_out(input_features=self.categorical_features)
        X_encoded_categorical_df = pd.DataFrame(X_encoded_categorical, index=X.index, columns=feature_names)

        # Combine numerical features with the encoded categorical features
        X_numerical = X.drop(columns=self.categorical_features)
        X_encoded = pd.concat([X_numerical, X_encoded_categorical_df], axis=1)
        predictions[name] = estimator.predict(X_encoded)
  # Perform majority voting to get the final predictions
  final_predictions_encoded = [
      Counter([predictions[name][i] for name in predictions]).most_common(1)[0][0]
      for i in range(len(X))]
  return self.label_encoders_['xgb'].inverse_transform(final_predictions_encoded)


In [ ]:
# Download KDDTrain+.txt from https://www.kaggle.com/datasets/hassan06/nslkdd/data?select=KDDTrain%2B.txt
# Load data
df_0 = pd.read_csv("KDDTrain+.txt")
df = df_0.copy()

# Set Column Names
columns = (['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
            'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
            'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
            'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
            'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
            'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
            'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
            'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack', 'level'])
df.columns = columns

FileNotFoundError: [Errno 2] No such file or directory: 'KDDTrain+.txt'

In [ ]:
df.info()

NameError: name 'df' is not defined

In [ ]:
# Encode traffic as normal or attack
attack_n = []
for i in df.attack:
    if i == 'normal':
        attack_n.append("normal")
    else:
        attack_n.append("attack")
df['attack'] = attack_n
X = df.drop(['attack', 'level'], axis=1)
y = df['attack']
categorical_features = ['protocol_type', 'service', 'flag']

NameError: name 'df' is not defined

In [ ]:
# Initialize the ensemble model
xgb_model = xgb.XGBClassifier(random_state=42)
catboost_model = CatBoostClassifier(random_state=42, verbose=0)
lgbm_model = lgb.LGBMClassifier(random_state=42)
ensemble_model = EnsembleClassifier(
    estimators=[('xgb', xgb_model), ('catboost', catboost_model), ('lgbm', lgbm_model)],
    categorical_features=categorical_features
)

In [ ]:
# Perform Stratified K-Fold Cross-Validation
n_splits = 5  # You can adjust the number of folds
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

accuracy_scores = []
classification_reports = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}/{n_splits}")
    X_train_fold, X_val_fold = X.iloc[train_index].copy(), X.iloc[val_index].copy()
    y_train_fold, y_val_fold = y.iloc[train_index].copy(), y.iloc[val_index].copy()

    # Train the ensemble on the current fold's training data
    ensemble_model.fit(X_train_fold, y_train_fold)

    # Make predictions on the validation set of the current fold
    y_pred_fold = ensemble_model.predict(X_val_fold)

    # Evaluate the predictions
    accuracy = accuracy_score(y_val_fold, y_pred_fold)
    report = classification_report(y_val_fold, y_pred_fold)

    accuracy_scores.append(accuracy)
    classification_reports.append(report)

# Print the results for each fold
print("\nCross-Validation Results:")
for i, acc in enumerate(accuracy_scores):
    print(f"Fold {i+1} Accuracy: {acc:.4f}")
    print(f"Fold {i+1} Classification Report:\n{classification_reports[i]}")
    print("-" * 30)

# Print the average accuracy across all folds
print(f"\nAverage Cross-Validation Accuracy: {np.mean(accuracy_scores):.4f}")

Fold 1/5
[LightGBM] [Info] Number of positive: 53873, number of negative: 46904
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061219 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3256
[LightGBM] [Info] Number of data points in the train set: 100777, number of used features: 110
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.534576 -> initscore=0.138526
[LightGBM] [Info] Start training from score 0.138526
Fold 2/5
[LightGBM] [Info] Number of positive: 53873, number of negative: 46904
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053083 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3265
[LightGBM] [Info] Number of data points in the train set: 100777, number of used features: 11